# Gerador de Instâncias Sintéticas
## Dimensionamento de Lotes com Subperíodos Diários

Este notebook gera instâncias sintéticas compatíveis com o solver MILP em
`optimization/solver.py`. Cada instância é gravada em `data/` como arquivo `.xlsx`.

### Estrutura do arquivo gerado

| Aba | Conteúdo |
|-----|----------|
| `Produtividade` | Taxa de produção (kg/h) por par (produto, máquina) |
| `Demanda` | Previsão de demanda mensal (kg) por produto |
| `Estoque` | Saldo de estoque no período anterior ao início do horizonte |
| `Custos` | Custo unitário (R$/kg) por produto |
| `Disponibilidade de maquinas` | Fração de dias disponíveis por máquina e mês |

### Como usar no sistema

1. Abra a interface — `python frontend/app.py` → `http://127.0.0.1:5000`
2. Selecione o arquivo gerado no seletor de *data file*
3. Configure `start_period`, `end_period`, `active_machines` e `coverage_months`
   conforme indicado no resumo impresso pela célula de geração
4. Execute a otimização


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


## Fatores de Complexidade

O número de variáveis binárias — principal driver de dificuldade do MILP — cresce com:

**2 x |I| x |J| x density x |T| x 30** (S_state e Delta_Setup) + **|J| x |T| x 30** (Z_day)

| Parâmetro | Facilita resolução | Dificulta resolução |
|-----------|--------------------|---------------------|
| `N_PRODUCTS`, `N_MACHINES`, `N_PERIODS` | valores pequenos | valores grandes |
| `PROD_DENSITY` | alto — mais opções de alocação | baixo — alocação restrita |
| `DEMAND_LEVEL` | < 0.7 — capacidade folgada | > 0.9 — déficit garantido |
| `DEMAND_CV` | 0 — demanda estável | alto — picos imprevisíveis |
| `HAS_SEASONALITY` | `False` | `True` com amplitude alta |
| `INITIAL_COVERAGE` | alto — buffer inicial grande | 0 — sem estoque inicial |
| `N_STOPS` / `AVAIL_MEAN` | sem paradas (`N_STOPS=0`) | muitas paradas, disponibilidade baixa |

> **Nota:** o estoque de segurança (α = `coverage_months`) é parâmetro do otimizador, configurado em `config/scenario.json` — não faz parte da instância Excel.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARÂMETROS DA INSTÂNCIA
# Edite este bloco para controlar dimensão e complexidade.
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42   # semente para reprodutibilidade

# ── Dimensões do problema ─────────────────────────────────────────────────────
N_PRODUCTS = 5    # |I| — SKUs distintos
N_MACHINES = 8    # |J| — máquinas
N_PERIODS  = 6    # |T| — meses no horizonte de planejamento
START_DATE = pd.Timestamp('2024-01-01')

# ── Capacidade (deve refletir config/capacity.json) ───────────────────────────
SHIFTS_PER_DAY  = 3   # turnos por dia
HOURS_PER_SHIFT = 8   # horas por turno  →  24 h/dia efetivos
DAYS_PER_WEEK   = 7   # dias por semana  →  round(7 x 4.33) = 30 dias/período

# ── Matriz de produtividade ───────────────────────────────────────────────────
# PROD_DENSITY: fração de pares (produto, máquina) compatíveis em [0, 1].
# Baixo → alocação mais restrita; Alto → mais opções para o solver.
PROD_DENSITY  = 0.60   # [0, 1]
PROD_RATE_MIN = 20.0   # kg/h — taxa mínima nos pares compatíveis
PROD_RATE_MAX = 80.0   # kg/h — taxa máxima nos pares compatíveis

# ── Demanda ───────────────────────────────────────────────────────────────────
# DEMAND_LEVEL: razão entre demanda total e capacidade máxima teórica.
#   < 0.7  →  capacidade folgada         (mais fácil)
#   0.7–0.9 → capacidade tensionada      (moderado)
#   > 1.0  →  déficit garantido          (mais difícil, haverá vendas perdidas)
DEMAND_LEVEL    = 0.70   # [0, inf)
DEMAND_CV       = 0.20   # coeficiente de variação entre períodos [0, 1]
HAS_SEASONALITY = True   # padrão senoidal com ciclo de 12 meses
SEASONALITY_AMP = 0.30   # amplitude relativa da sazonalidade [0, 1]

# ── Estoque inicial ───────────────────────────────────────────────────────────
# Quantos meses de demanda média o estoque inicial cobre.
# 0 = sem estoque → pressão máxima no início do horizonte.
INITIAL_COVERAGE = 1   # meses

# ── Custos unitários ──────────────────────────────────────────────────────────
COST_MIN = 5.0    # R$/kg
COST_MAX = 20.0   # R$/kg

# ── Setup ─────────────────────────────────────────────────────────────────────
# As N_HIGH_SETUP primeiras máquinas recebem tempo de setup alto.
N_HIGH_SETUP    = 1    # número de máquinas com setup longo
SETUP_TIME_HIGH = 7.0  # horas de setup nas máquinas pesadas
SETUP_TIME_LOW  = 3.0  # horas de setup nas máquinas comuns

# ── Disponibilidade das máquinas ──────────────────────────────────────────────
# N_STOPS: número de períodos com parada programada por máquina.
# AVAIL_MEAN: disponibilidade média nesses períodos [0.5, 1.0].
# N_STOPS = 0 → todas as máquinas sempre disponíveis.
N_STOPS    = 1      # períodos com parada por máquina
AVAIL_MEAN = 0.90   # disponibilidade nos períodos com parada

# ── Arquivo de saída ──────────────────────────────────────────────────────────
OUTPUT_NAME = f'instance_P{N_PRODUCTS}_M{N_MACHINES}_T{N_PERIODS}.xlsx'
OUTPUT_PATH = Path('data') / OUTPUT_NAME


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNÇÕES DE GERAÇÃO
# ─────────────────────────────────────────────────────────────────────────────

def make_products(n):
    '''Cria n SKUs sintéticos no formato (MODELO, TIPO).'''
    return [(f'P{i+1:02d}', 'LISO') for i in range(n)]


def make_machine_ids(n):
    '''Cria n IDs de máquinas como strings inteiras sequenciais.'''
    return [str(j + 1) for j in range(n)]


def make_periods(start, n):
    '''Gera n períodos mensais no formato MM/AAAA.'''
    return [(start + pd.DateOffset(months=t)).strftime('%m/%Y') for t in range(n)]


def make_productivity(products, machines, density, rate_min, rate_max, rng):
    '''
    Gera a matriz de produtividade (kg/h) com esparsidade controlada.

    Cada par (produto, máquina) é compatível com probabilidade = density.
    Garante que todo produto tenha ao menos uma máquina compatível.
    Pares incompatíveis ficam como NaN — ignorados pelo sistema.
    '''
    n_p, n_m = len(products), len(machines)

    compat = rng.random((n_p, n_m)) < density
    for i in range(n_p):
        if not compat[i].any():
            compat[i, rng.integers(n_m)] = True

    rates = rng.uniform(rate_min, rate_max, (n_p, n_m))
    rates[~compat] = np.nan
    return rates


def make_demand(n_products, n_periods, n_machines, rates,
                hours_per_day, days_per_period,
                demand_level, cv, has_season, amplitude, rng):
    '''
    Gera demanda mensal por produto (kg).

    A demanda total é escalonada por demand_level em relação à capacidade
    produtiva máxima teórica: todas as máquinas rodando continuamente
    (n_machines x hours_per_day x days_per_period x avg_rate).
    '''
    avg_rate = float(np.nanmean(rates))
    total_capacity = n_machines * hours_per_day * days_per_period * avg_rate

    base_per_product = total_capacity * demand_level / n_products

    month_idx = np.arange(n_periods)
    seasonal = (1 + amplitude * np.sin(2 * np.pi * month_idx / 12)
                if has_season else np.ones(n_periods))

    demand = np.zeros((n_products, n_periods))
    for i in range(n_products):
        noise = rng.normal(1.0, cv, n_periods).clip(min=0.1)
        demand[i] = base_per_product * seasonal * noise

    return demand


def make_costs(n_products, cost_min, cost_max, rng):
    '''Gera custos unitários por produto (R$/kg).'''
    return rng.uniform(cost_min, cost_max, n_products)


def make_initial_inventory(demand, coverage_months):
    '''
    Calcula o estoque inicial como múltiplo da demanda média por produto.

    coverage_months = 0  → sem estoque inicial.
    coverage_months = 2  → estoque cobre 2 meses de demanda média.
    '''
    return demand.mean(axis=1) * coverage_months


def make_availability(n_machines, n_periods, avail_mean, n_stops, rng):
    '''
    Gera disponibilidade mensal das máquinas como fração de dias úteis ativos.

    Para cada máquina, sorteia n_stops períodos com disponibilidade reduzida.
    Os demais períodos têm disponibilidade plena (1.0).
    '''
    avail = np.ones((n_machines, n_periods))

    if n_stops > 0 and n_periods > 0:
        for j in range(n_machines):
            stop_idx = rng.choice(n_periods, size=min(n_stops, n_periods), replace=False)
            for t in stop_idx:
                low  = max(0.50, avail_mean - 0.20)
                high = min(0.99, avail_mean + 0.05)
                avail[j, t] = rng.uniform(low, high)

    return avail


In [ ]:
def save_instance(path, products, machines, periods, start_date,
                  rates, demand, costs, initial_inv, avail):
    '''
    Grava a instância no formato Excel compatível com o sistema de otimização.

    Produtividade, Demanda e Estoque usam startrow=1 para que o pandas leia
    os cabeçalhos na linha correta (header=1 em processing/data.py).
    Custos e Disponibilidade usam o padrão (header=0).
    '''
    path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(path, engine='openpyxl') as writer:

        # ── Produtividade ──────────────────────────────────────────────────
        prod_data = {
            'MODELO': [p[0] for p in products],
            'TIPO':   [p[1] for p in products],
        }
        for j, m in enumerate(machines):
            prod_data[m] = [rates[i, j] for i in range(len(products))]
        pd.DataFrame(prod_data).to_excel(writer, sheet_name='Produtividade', startrow=1, index=False)

        # ── Demanda ────────────────────────────────────────────────────────
        dem_data = {
            'MODELO': [p[0] for p in products],
            'TIPO':   [p[1] for p in products],
        }
        for t, per in enumerate(periods):
            dem_data[per] = demand[:, t]
        pd.DataFrame(dem_data).to_excel(writer, sheet_name='Demanda', startrow=1, index=False)

        # ── Estoque ────────────────────────────────────────────────────────
        # Uma coluna pré-horizonte com o saldo inicial de cada produto.
        # O sistema usa o último saldo com data <= start_period como estoque inicial.
        pre_period = (start_date - pd.DateOffset(months=1)).strftime('%m/%Y')
        inv_data = {
            'MODELO':   [p[0] for p in products],
            'TIPO':     [p[1] for p in products],
            pre_period: initial_inv,
        }
        pd.DataFrame(inv_data).to_excel(writer, sheet_name='Estoque', startrow=1, index=False)

        # ── Custos ─────────────────────────────────────────────────────────
        cost_data = {
            'MODELO':         [p[0] for p in products],
            'TIPO':           [p[1] for p in products],
            'CUSTO_UNITARIO': costs,
        }
        pd.DataFrame(cost_data).to_excel(writer, sheet_name='Custos', index=False)

        # ── Disponibilidade de máquinas ────────────────────────────────────
        avail_data = {'DATA': periods}
        for j, m in enumerate(machines):
            avail_data[m] = avail[j, :]
        pd.DataFrame(avail_data).to_excel(writer, sheet_name='Disponibilidade de maquinas', index=False)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# GERAR E SALVAR INSTÂNCIA
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(SEED)
hours_per_day  = SHIFTS_PER_DAY * HOURS_PER_SHIFT
days_per_period = round(DAYS_PER_WEEK * 4.33)   # mesma lógica do planner.py

# Estrutura
products = make_products(N_PRODUCTS)
machines = make_machine_ids(N_MACHINES)
periods  = make_periods(START_DATE, N_PERIODS)

# Dados
rates    = make_productivity(products, machines, PROD_DENSITY,
                             PROD_RATE_MIN, PROD_RATE_MAX, rng)
demand   = make_demand(N_PRODUCTS, N_PERIODS, N_MACHINES, rates,
                       hours_per_day, days_per_period,
                       DEMAND_LEVEL, DEMAND_CV, HAS_SEASONALITY, SEASONALITY_AMP, rng)
costs    = make_costs(N_PRODUCTS, COST_MIN, COST_MAX, rng)
init_inv = make_initial_inventory(demand, INITIAL_COVERAGE)
avail    = make_availability(N_MACHINES, N_PERIODS, AVAIL_MEAN, N_STOPS, rng)

save_instance(OUTPUT_PATH, products, machines, periods, START_DATE,
              rates, demand, costs, init_inv, avail)

# ── Resumo ────────────────────────────────────────────────────────────────────
n_compat = int(np.sum(~np.isnan(rates)))
n_days   = N_PERIODS * days_per_period
n_binary = 2 * n_compat * n_days + N_MACHINES * n_days
n_cont   = 2 * N_PRODUCTS * N_PERIODS

capacity     = N_MACHINES * hours_per_day * days_per_period * float(np.nanmean(rates))
avg_demand_t = float(demand.sum(axis=0).mean())
high_mach    = machines[:N_HIGH_SETUP]
end_str      = (START_DATE + pd.DateOffset(months=N_PERIODS - 1)).strftime('%Y-%m-%d')

sep = '-' * 58
print(sep)
print(f'  Arquivo : {OUTPUT_NAME}')
print(sep)
print(f'  Produtos    : {N_PRODUCTS}   ({", ".join(p[0] for p in products)})')
print(f'  Maquinas    : {N_MACHINES}   (1 ate {N_MACHINES})')
print(f'  Periodos    : {N_PERIODS}   ({periods[0]} ate {periods[-1]})')
print(f'  Pares compat: {n_compat} / {N_PRODUCTS * N_MACHINES}  ({n_compat / (N_PRODUCTS * N_MACHINES):.0%})')
print(sep)
print(f'  Variaveis binarias  : {n_binary:>10,}')
print(f'  Variaveis continuas : {n_cont:>10,}')
print(f'  Total               : {n_binary + n_cont:>10,}')
print(sep)
print(f'  Capacidade/periodo  : {capacity:>12,.0f} kg')
print(f'  Demanda media/per.  : {avg_demand_t:>12,.0f} kg')
print(f'  Taxa de carga       : {avg_demand_t / capacity:>12.1%}')
print(sep)
print(f'  Estoque inicial      : {INITIAL_COVERAGE} mes(es) de cobertura')
print(sep)
print(f'  Configuracao sugerida (config/)')
print(f'    start_period        : {START_DATE.strftime("%Y-%m-%d")} 00:00:00')
print(f'    end_period          : {end_str} 00:00:00')
print(f'    active_machines     : {machines}')
print(f'    high_setup_machines : {high_mach}')
print(f'    setup_time_high     : {SETUP_TIME_HIGH}')
print(f'    setup_time_low      : {SETUP_TIME_LOW}')
print(sep)


---

## Instâncias Pré-configuradas

Os perfis abaixo cobrem diferentes pontos do espectro de complexidade.
Execute a célula seguinte para gerar todos os arquivos de uma vez em `data/`.

| Perfil | Produtos | Máquinas | Períodos | Vars binárias (aprox.) | Característica |
|--------|----------|----------|----------|------------------------|----------------|
| `micro` | 2 | 3 | 2 | ~720 | teste de viabilidade |
| `pequeno` | 4 | 6 | 4 | ~8.640 | resolve em segundos |
| `medio` | 8 | 12 | 6 | ~62.208 | resolve em ~1 min |
| `grande` | 12 | 20 | 12 | ~414.720 | resolve em vários minutos |
| `real` | 17 | 28 | 13 | ~1.270.080 | escala do caso Riberball |


In [ ]:
PROFILES = {
    'micro': dict(
        description='Teste de viabilidade — resolve em < 5s',
        N_PRODUCTS=2,  N_MACHINES=3,  N_PERIODS=2,
        PROD_DENSITY=0.90, PROD_RATE_MIN=20.0, PROD_RATE_MAX=80.0,
        DEMAND_LEVEL=0.50, DEMAND_CV=0.10, HAS_SEASONALITY=False, SEASONALITY_AMP=0.00,
        INITIAL_COVERAGE=1, COST_MIN=10.0, COST_MAX=10.0,
        N_HIGH_SETUP=0, SETUP_TIME_HIGH=3.0, SETUP_TIME_LOW=3.0,
        N_STOPS=0, AVAIL_MEAN=1.00,
    ),
    'pequeno': dict(
        description='Instancia pequena — resolve em < 30s',
        N_PRODUCTS=4,  N_MACHINES=6,  N_PERIODS=4,
        PROD_DENSITY=0.70, PROD_RATE_MIN=20.0, PROD_RATE_MAX=80.0,
        DEMAND_LEVEL=0.60, DEMAND_CV=0.15, HAS_SEASONALITY=False, SEASONALITY_AMP=0.00,
        INITIAL_COVERAGE=1, COST_MIN=5.0,  COST_MAX=20.0,
        N_HIGH_SETUP=0, SETUP_TIME_HIGH=7.0, SETUP_TIME_LOW=3.0,
        N_STOPS=0, AVAIL_MEAN=1.00,
    ),
    'medio': dict(
        description='Instancia media — resolve em ~1 min',
        N_PRODUCTS=8,  N_MACHINES=12, N_PERIODS=6,
        PROD_DENSITY=0.50, PROD_RATE_MIN=20.0, PROD_RATE_MAX=80.0,
        DEMAND_LEVEL=0.75, DEMAND_CV=0.20, HAS_SEASONALITY=True,  SEASONALITY_AMP=0.25,
        INITIAL_COVERAGE=1, COST_MIN=5.0,  COST_MAX=20.0,
        N_HIGH_SETUP=1, SETUP_TIME_HIGH=7.0, SETUP_TIME_LOW=3.0,
        N_STOPS=1, AVAIL_MEAN=0.90,
    ),
    'grande': dict(
        description='Instancia grande — resolve em varios minutos',
        N_PRODUCTS=12, N_MACHINES=20, N_PERIODS=12,
        PROD_DENSITY=0.40, PROD_RATE_MIN=20.0, PROD_RATE_MAX=80.0,
        DEMAND_LEVEL=0.85, DEMAND_CV=0.25, HAS_SEASONALITY=True,  SEASONALITY_AMP=0.35,
        INITIAL_COVERAGE=0, COST_MIN=5.0,  COST_MAX=30.0,
        N_HIGH_SETUP=3, SETUP_TIME_HIGH=7.0, SETUP_TIME_LOW=3.0,
        N_STOPS=2, AVAIL_MEAN=0.85,
    ),
    'real': dict(
        description='Dimensao comparavel ao caso Riberball (28 maq., 17 prod., 13 meses)',
        N_PRODUCTS=17, N_MACHINES=28, N_PERIODS=13,
        PROD_DENSITY=0.40, PROD_RATE_MIN=20.0, PROD_RATE_MAX=80.0,
        DEMAND_LEVEL=0.90, DEMAND_CV=0.30, HAS_SEASONALITY=True,  SEASONALITY_AMP=0.40,
        INITIAL_COVERAGE=0, COST_MIN=5.0,  COST_MAX=30.0,
        N_HIGH_SETUP=2, SETUP_TIME_HIGH=7.0, SETUP_TIME_LOW=3.0,
        N_STOPS=2, AVAIL_MEAN=0.80,
    ),
}


def generate_from_profile(name, profile, seed=42):
    '''Gera e salva uma instância a partir de um perfil pré-configurado.'''
    rng_p   = np.random.default_rng(seed)
    n_p, n_m, n_t = profile['N_PRODUCTS'], profile['N_MACHINES'], profile['N_PERIODS']
    start   = pd.Timestamp('2024-01-01')
    h_day   = 3 * 8          # shifts x hours — fixo conforme capacity.json
    d_per   = round(7 * 4.33)  # days_per_week x 4.33 — mesma lógica do planner.py

    products_p = make_products(n_p)
    machines_p = make_machine_ids(n_m)
    periods_p  = make_periods(start, n_t)

    rates_p  = make_productivity(products_p, machines_p, profile['PROD_DENSITY'],
                                 profile['PROD_RATE_MIN'], profile['PROD_RATE_MAX'], rng_p)
    demand_p = make_demand(n_p, n_t, n_m, rates_p, h_day, d_per,
                           profile['DEMAND_LEVEL'], profile['DEMAND_CV'],
                           profile['HAS_SEASONALITY'], profile['SEASONALITY_AMP'], rng_p)
    costs_p  = make_costs(n_p, profile['COST_MIN'], profile['COST_MAX'], rng_p)
    inv_p    = make_initial_inventory(demand_p, profile['INITIAL_COVERAGE'])
    avail_p  = make_availability(n_m, n_t, profile['AVAIL_MEAN'], profile['N_STOPS'], rng_p)

    fname = f'instance_{name}_P{n_p}_M{n_m}_T{n_t}.xlsx'
    save_instance(Path('data') / fname, products_p, machines_p, periods_p, start,
                  rates_p, demand_p, costs_p, inv_p, avail_p)

    n_compat_p = int(np.sum(~np.isnan(rates_p)))
    n_binary_p = 2 * n_compat_p * n_t * d_per + n_m * n_t * d_per

    return {
        'Perfil':        name,
        'Descricao':     profile['description'],
        'Produtos':      n_p,
        'Maquinas':      n_m,
        'Periodos':      n_t,
        'Pares compat.': n_compat_p,
        'Vars binarias': f'{n_binary_p:,}',
        'Arquivo':       fname,
    }


# Gera todos os perfis de uma vez
rows = []
for profile_name, profile_params in PROFILES.items():
    row = generate_from_profile(profile_name, profile_params, seed=42)
    rows.append(row)
    print(f'  OK  {profile_name:10s}  —  {profile_params["description"]}')

print()
pd.DataFrame(rows).set_index('Perfil')
